# MLP erstellen (Betonstärkenprädiktion)

In [1]:
import kagglehub
import pandas
import matplotlib.pyplot as plt
import numpy

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

import torch
from torch import nn

# 1. Daten einlesen
path = kagglehub.dataset_download("mchilamwar/predict-concrete-strength")
df = pandas.read_csv(f"{path}/ConcreteStrengthData.csv")
df = df.rename( {"CementComponent " : "CementComponent"}, axis="columns" )

# 2. Was ist Input? Was ist Output?
#    Train-Test-Split
target = "Strength"
x = df.drop( target, axis="columns" )
y = df[ [target] ]
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2,
                                                    random_state=42)

# 3. Skalierung der Daten (Achtung vor Data Leakage!)
x_scaler = RobustScaler()
y_scaler = RobustScaler()
x_train_scaled = x_scaler.fit_transform(x_train)
y_train_scaled = y_scaler.fit_transform(y_train)

# 4. Daten als Tensoren "verpacken"
x_train_scaled_tensor = torch.tensor(x_train_scaled, dtype=torch.float32)
y_train_scaled_tensor = torch.tensor(y_train_scaled, dtype=torch.float32)

# 5. MLP bauen
N1 = 128
N2 = 64
nr_inputs = len(x_train.columns)
nr_outputs = len(y_train.columns)
model = nn.Sequential(
            nn.Linear(nr_inputs, N1),
            nn.ReLU(),

            nn.Linear(N1, N2),
            nn.ReLU(),
    
            nn.Linear(N2, nr_outputs)
)

# 6. Jetzt das MLP trainieren
model.train()
loss_function = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
for epoch in range(1000):
    predictions = model(x_train_scaled_tensor)
    loss = loss_function(predictions, y_train_scaled_tensor)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# 7. Jetzt das MLP testen
model.eval()
x_test_scaled = x_scaler.transform(x_test)
x_test_scaled_tensor = torch.tensor(x_test_scaled, dtype=torch.float32)
with torch.no_grad():
    predictions_test_scaled_tensor = model(x_test_scaled_tensor)
predictions_test = y_scaler.inverse_transform( predictions_test_scaled_tensor )

# 8. Wie gut ist das MLP?
y_pred = predictions_test.flatten()
y_true = y_test[target].values
MAE = mean_absolute_error(y_true, y_pred)
MAPE = mean_absolute_percentage_error(y_true, y_pred)*100
print(f"MAE={MAE:.2f} [Pascal], MAPE={MAPE:.2f}%")

MAE=3.14 [Pascal], MAPE=9.93%


# MLP inklusive Skalierung als ONNX exportieren

Zuerst die Trainingszelle oben ausführen. Wir speichern die gesamte Berechnung in **einer Datei**:

**Rohdaten → Eingabeskalierung → MLP → Rückskalierung → Betonfestigkeit**

Für die hier verwendeten `RobustScaler` sind das einfache Rechenoperationen:
- Eingaben skalieren: `(x - x_scaler.center_) / x_scaler.scale_`
- Ausgabe zurückskalieren: `y_scaled * y_scaler.scale_ + y_scaler.center_`

Die bereits gelernten Skalierungswerte werden als Konstanten mit exportiert. Es wird nichts erneut trainiert oder angepasst.

Benötigt werden `onnx` und `onnxruntime` (in `env_teaching` bereits vorhanden). Falls sie in einer anderen Umgebung fehlen: `%pip install onnx onnxruntime`.

In [2]:
class MLPMitSkalierung(nn.Module):
    def __init__(self, mlp, x_scaler, y_scaler):
        super().__init__()
        self.mlp = mlp
        # Buffer sind feste Werte, keine trainierbaren Parameter.
        self.register_buffer("x_center", torch.tensor(x_scaler.center_, dtype=torch.float32))
        self.register_buffer("x_scale", torch.tensor(x_scaler.scale_, dtype=torch.float32))
        self.register_buffer("y_center", torch.tensor(y_scaler.center_, dtype=torch.float32))
        self.register_buffer("y_scale", torch.tensor(y_scaler.scale_, dtype=torch.float32))

    def forward(self, x):
        x_scaled = (x - self.x_center) / self.x_scale
        y_scaled = self.mlp(x_scaled)
        return y_scaled * self.y_scale + self.y_center


export_model = MLPMitSkalierung(model, x_scaler, y_scaler).eval()

In [3]:
type(export_model)

__main__.MLPMitSkalierung

In [9]:
%pip install onnxscript

  Using cached onnxscript-0.7.2-py3-none-any.whl.metadata (13 kB)
Using cached onnxscript-0.7.2-py3-none-any.whl (754 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [onnxscript]2 [onnxscript]
Note: you may need to restart the kernel to use updated packages.


In [10]:
# Zwei Beispielzeilen für den Export mit variabler Batchgröße
beispiel = torch.tensor(
    x_train.iloc[:2].to_numpy(), dtype=torch.float32
)

torch.onnx.export(
    export_model,
    (beispiel,),
    "beton_mlp.onnx",
    input_names=["eingaben"],
    output_names=["betonfestigkeit"],
    dynamic_shapes={"x": {0: torch.export.Dim("anzahl")}},
    opset_version=18,
    dynamo=True,
    external_data=False,
)

[torch.onnx] Obtain model graph for `MLPMitSkalierung([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MLPMitSkalierung([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 18},
            producer_name='pytorch',
            producer_version='2.9.1+cu128',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"eingaben"<FLOAT,[anzahl,8]>
            ),
            outputs=(
                %"betonfestigkeit"<FLOAT,[anzahl,1]>
            ),
            initializers=(
                %"mlp.0.weight"<FLOAT,[128,8]>{TorchTensor(...)},
                %"mlp.0.bias"<FLOAT,[128]>{TorchTensor(...)},
                %"mlp.2.weight"<FLOAT,[64,128]>{TorchTensor(...)},
                %"mlp.2.bias"<FLOAT,[64]>{TorchTensor(...)},
                %"mlp.4.weight"<FLOAT,[1,64]>{TorchTensor(...)},
                %"mlp.4.bias"<FLOAT,[1]>{TorchTensor<FLOAT,[1]>(Parameter containing: tensor([0.1139], requires_grad=True), name='mlp.4.bias')},
                %"x_center"<FLOAT,[8]>{TorchTe

"x" bezeichnet das Argument von forward(self, x). external_data=False sorgt dafür, dass die Gewichte direkt in unserer einzelnen ONNX-Datei gespeichert werden.

In [11]:
!ls -la

total 84
drwxrwxr-x  2 juebrauer juebrauer  4096 Oct 21 08:08 .
drwxrwxr-x 77 juebrauer juebrauer  4096 Oct 21 08:23 ..
-rw-rw-r--  1 juebrauer juebrauer 49148 Oct 21 08:25 beton_mlp.onnx
-rw-rw-r--  1 juebrauer juebrauer 24723 Oct 21 08:27 onnx_beispiele.ipynb


# ONNX-Datei laden und neue Daten vorhersagen

**Dieser Abschnitt funktioniert auch nach einem Kernel-Neustart**, sobald `beton_mlp.onnx` im Arbeitsverzeichnis liegt. Für die Vorhersage benötigen wir nur ONNX Runtime, NumPy und hier Pandas zur Tabellenanzeige.

Die zwei Zeilen unten sind beispielhafte Betonmischungen, deren Werte du ändern kannst. Wichtig: **dieselbe Spaltenreihenfolge wie beim Training**, `float32` und eine zweidimensionale Eingabe `(Anzahl Zeilen, 8)` – auch bei nur einer Mischung. Die Daten werden **unskaliert** übergeben; beide Skalierungen stecken bereits in der Datei.

In [12]:
import numpy as np
import pandas as pd
import onnxruntime as ort

session = ort.InferenceSession("beton_mlp.onnx", providers=["CPUExecutionProvider"])

# Reihenfolge wie beim Training: sieben Mischungsmerkmale, dann Alter in Tagen.
neue_daten = pd.DataFrame(
    [
        [300.0, 50.0, 20.0, 180.0, 5.0, 1000.0, 750.0, 28.0],
        [350.0, 40.0, 30.0, 170.0, 6.0, 1050.0, 700.0, 56.0],
    ],
    columns=[
        "CementComponent", "BlastFurnaceSlag", "FlyAshComponent",
        "WaterComponent", "SuperplasticizerComponent",
        "CoarseAggregateComponent", "FineAggregateComponent", "AgeInDays",
    ],
)

# run() liefert eine Liste der Modellausgaben; unser Modell hat genau eine.
eingaben = neue_daten.to_numpy(dtype=np.float32)
praediktionen = session.run(None, {"eingaben": eingaben})[0]

ergebnis = neue_daten.copy()
ergebnis["Vorhergesagte Betonfestigkeit"] = praediktionen.ravel()
ergebnis

,CementComponent,BlastFurnaceSlag,FlyAshComponent,WaterComponent,SuperplasticizerComponent,CoarseAggregateComponent,FineAggregateComponent,AgeInDays,Vorhergesagte Betonfestigkeit
0,300.0,50.0,20.0,180.0,5.0,1000.0,750.0,28.0,36.761852
1,350.0,40.0,30.0,170.0,6.0,1050.0,700.0,56.0,52.509460


Die Vorhersagen sind bereits in der ursprünglichen Einheit der Zielspalte `Strength`. Für diese neuen Mischungen sind keine tatsächlichen Zielwerte bekannt; die Tabelle zeigt Modellvorhersagen.

Dokumentation: [ONNX Runtime – Python API](https://onnxruntime.ai/docs/api/python/api_summary.html).

# Optional: Stimmen PyTorch und ONNX überein?

Diese Kontrollzelle benötigt zusätzlich die Trainingszelle im selben Kernel. Wir vergleichen auf den Testdaten die bisherige Berechnung (Scikit-learn-Skalierer + PyTorch) mit der ONNX-Datei. Kleine Abweichungen durch `float32` und unterschiedliche Rechenwege sind normal.

In [13]:
onnx_test = session.run(None, {"eingaben": x_test.to_numpy(dtype=np.float32)})[0]

max_abweichung = np.max(np.abs(predictions_test - onnx_test))
print(f"Maximale absolute Abweichung: {max_abweichung:.6f}")
np.testing.assert_allclose(onnx_test, predictions_test, rtol=1e-4, atol=1e-4)
print("PyTorch und ONNX liefern praktisch gleiche Vorhersagen.")

Maximale absolute Abweichung: 0.000021
PyTorch und ONNX liefern praktisch gleiche Vorhersagen.


atol erlaubt eine kleine absolute Abweichung. Das ist besonders bei Werten nahe null wichtig.

rtol erlaubt eine Abweichung proportional zur Größe des Referenzwerts.

# Dasselbe Problem mit einem Random Forest

Wir verwenden dieselben Trainings- und Testdaten (`x_train`, `x_test`, `y_train`, `y_test`) aus der ersten Trainingszelle. Ein `RandomForestRegressor` mittelt die Vorhersagen vieler Entscheidungsbäume. Hier benötigen wir **keine Skalierung** der Eingaben oder Zielwerte.

Für den ONNX-Export von Scikit-learn-Modellen verwenden wir `skl2onnx`. Falls das Paket fehlt: `%pip install skl2onnx`.

## 1. Trainieren und testen

In [14]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(x_train, y_train["Strength"])  # Eine Zielspalte als eindimensionaler Vektor

rf_test = rf.predict(x_test)
print(f"MAE: {mean_absolute_error(y_test['Strength'], rf_test):.2f}")
print(f"MAPE: {mean_absolute_percentage_error(y_test['Strength'], rf_test):.2%}")

MAE: 3.74
MAPE: 12.32%


## 2. Random Forest als ONNX exportieren

Eine Beispielzeile legt Eingabetyp und Anzahl der Merkmale fest. Das exportierte Modell kann anschließend beliebig viele Zeilen verarbeiten. Alle Bäume werden in `beton_rf.onnx` gespeichert.

Dokumentation: [sklearn-onnx – Einführung](https://onnx.ai/sklearn-onnx/introduction.html).

In [15]:
import numpy as np
from skl2onnx import to_onnx

rf_onnx = to_onnx(
    rf,
    x_train.iloc[:1].to_numpy(dtype=np.float32),
    target_opset=18,
)
with open("beton_rf.onnx", "wb") as datei:
    datei.write(rf_onnx.SerializeToString())

print("Gespeichert: beton_rf.onnx")

Gespeichert: beton_rf.onnx


In [16]:
type(rf_onnx)

onnx.onnx_ml_pb2.ModelProto

## 3. ONNX-Datei laden und anwenden

Diese Zelle funktioniert **auch nach einem Kernel-Neustart**, wenn `beton_rf.onnx` im Arbeitsverzeichnis liegt. Der trainierte Python-Random-Forest wird dafür nicht benötigt.

Wir übergeben unskalierte Daten als `float32`, in derselben Spaltenreihenfolge wie beim Training. Die Ausgabe ist bereits in der ursprünglichen Einheit von `Strength`.

In [19]:
import numpy as np
import pandas as pd
import onnxruntime as ort

rf_session = ort.InferenceSession("beton_rf.onnx", providers=["CPUExecutionProvider"])

rf_neue_daten = pd.DataFrame(
    [
        [300.0, 50.0, 20.0, 180.0, 5.0, 1000.0, 750.0, 28.0],
        [350.0, 40.0, 30.0, 170.0, 6.0, 1050.0, 700.0, 56.0],
    ],
    columns=[
        "CementComponent", "BlastFurnaceSlag", "FlyAshComponent",
        "WaterComponent", "SuperplasticizerComponent",
        "CoarseAggregateComponent", "FineAggregateComponent", "AgeInDays",
    ],
)

input_name = rf_session.get_inputs()[0].name
rf_eingaben = rf_neue_daten.to_numpy(dtype=np.float32)
rf_praediktionen = rf_session.run(None, {input_name: rf_eingaben})[0]

rf_ergebnis = rf_neue_daten.copy()
rf_ergebnis["Vorhergesagte Betonfestigkeit"] = rf_praediktionen.ravel()
rf_ergebnis

,CementComponent,BlastFurnaceSlag,FlyAshComponent,WaterComponent,SuperplasticizerComponent,CoarseAggregateComponent,FineAggregateComponent,AgeInDays,Vorhergesagte Betonfestigkeit
0,300.0,50.0,20.0,180.0,5.0,1000.0,750.0,28.0,41.457813
1,350.0,40.0,30.0,170.0,6.0,1050.0,700.0,56.0,54.279018


In [20]:
input_name

'X'

In [21]:
rf_eingaben

array([[ 300.,   50.,   20.,  180.,    5., 1000.,  750.,   28.],
       [ 350.,   40.,   30.,  170.,    6., 1050.,  700.,   56.]],
      dtype=float32)

## Optional: Scikit-learn und ONNX vergleichen

Diese Kontrolle benötigt die RF-Trainingszelle im selben Kernel. `ravel()` bringt die ONNX-Ausgabe in dieselbe eindimensionale Form wie `rf.predict()`. Kleine Rundungsabweichungen sind normal.

In [22]:
rf_onnx_test = rf_session.run(
    None, {input_name: x_test.to_numpy(dtype=np.float32)}
)[0].ravel()

print(f"Maximale absolute Abweichung: {np.max(np.abs(rf_onnx_test - rf_test)):.6f}")
np.testing.assert_allclose(rf_onnx_test, rf_test, rtol=1e-4, atol=1e-4)
print("Scikit-learn und ONNX liefern praktisch gleiche Vorhersagen.")

Maximale absolute Abweichung: 0.000044
Scikit-learn und ONNX liefern praktisch gleiche Vorhersagen.
